In [1]:
!pip install torch torchvision

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(28*28, 128)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = x.view(-1, 28*28)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [3]:
transform = transforms.Compose([transforms.ToTensor()])
dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)

num_clients = 5
data_per_client = len(dataset) // num_clients

client_datasets = []
for i in range(num_clients):
    indices = list(range(i * data_per_client, (i+1) * data_per_client))
    client_datasets.append(Subset(dataset, indices))

100%|██████████| 9.91M/9.91M [08:21<00:00, 19.8kB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 29.9kB/s]
100%|██████████| 1.65M/1.65M [01:56<00:00, 14.1kB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 3.20MB/s]


In [4]:
def add_dp_noise(state_dict, noise_scale=0.01):
    noisy_state_dict = {}

    for key in state_dict:
        noise = torch.normal(mean=0, std=noise_scale, size=state_dict[key].shape)
        noisy_state_dict[key] = state_dict[key] + noise

    return noisy_state_dict

In [5]:
def train_local_dp(model, dataset, epochs=1, clip_value=1.0, noise_scale=0.01):
    model.train()
    loader = DataLoader(dataset, batch_size=32, shuffle=True)

    optimizer = optim.SGD(model.parameters(), lr=0.01)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        for data, target in loader:
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()

            # 🔹 Gradient Clipping (DP step)
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip_value)

            optimizer.step()

    # Get model updates
    local_weights = model.state_dict()

    # 🔹 Add DP Noise before sending to server
    noisy_weights = add_dp_noise(local_weights, noise_scale)

    return noisy_weights

In [6]:
def federated_averaging(global_model, client_updates):
    new_state_dict = global_model.state_dict()

    for key in new_state_dict.keys():
        new_state_dict[key] = torch.stack(
            [client_updates[i][key].float() for i in range(len(client_updates))],
            0
        ).mean(0)

    global_model.load_state_dict(new_state_dict)
    return global_model

In [7]:
global_model = SimpleNN()
rounds = 5

for r in range(rounds):
    print(f"Round {r+1}")

    client_updates = []

    for i in range(num_clients):
        local_model = SimpleNN()
        local_model.load_state_dict(global_model.state_dict())

        updated_weights = train_local_dp(
            local_model,
            client_datasets[i],
            epochs=1,
            clip_value=1.0,
            noise_scale=0.02   # controls privacy level
        )

        client_updates.append(updated_weights)

    # Server aggregates noisy updates
    global_model = federated_averaging(global_model, client_updates)

print("Federated Training with DP Complete!")

Round 1
Round 2
Round 3
Round 4
Round 5
Federated Training with DP Complete!
